# Challenge 1


In [3]:
import base64


def _to_b64(_in: bytes) -> bytes:
    return base64.b64encode(_in)


def hex_to_b64(s: str) -> str:
    return _to_b64(bytes.fromhex(s)).decode()


_input = "49276d206b696c6c696e6720796f757220627261696e206c696b65206120706f69736f6e6f7573206d757368726f6f6d"
assert (
    hex_to_b64(_input)
    == "SSdtIGtpbGxpbmcgeW91ciBicmFpbiBsaWtlIGEgcG9pc29ub3VzIG11c2hyb29t"
)

# Challenge 2


In [4]:
def _equal_length_xor(_in1: bytes, _in2: bytes) -> bytes:
    return bytes([a ^ b for a, b in zip(_in1, _in2, strict=True)])


def equal_length_xor(_hex1: str, _hex2: str) -> str:
    return _equal_length_xor(bytes.fromhex(_hex1), bytes.fromhex(_hex2)).hex()


_input = "1c0111001f010100061a024b53535009181c"
_input2 = "686974207468652062756c6c277320657965"

assert equal_length_xor(_input, _input2) == "746865206b696420646f6e277420706c6179"

# Challenge 3


In [5]:
freqs = {
    "a": 0.0855,
    "b": 0.0160,
    "c": 0.0316,
    "d": 0.0387,
    "e": 0.1209,
    "f": 0.0218,
    "g": 0.0209,
    "h": 0.0496,
    "i": 0.0732,
    "j": 0.0022,
    "k": 0.0081,
    "l": 0.0420,
    "m": 0.0253,
    "n": 0.0717,
    "o": 0.0747,
    "p": 0.0206,
    "q": 0.0010,
    "r": 0.0633,
    "s": 0.0673,
    "t": 0.0894,
    "u": 0.0268,
    "v": 0.0106,
    "w": 0.0182,
    "x": 0.0019,
    "y": 0.0172,
    "z": 0.0011,
}


def _single_char_xor(char: int, cypher: bytes) -> bytes:
    return bytes([char ^ b for b in cypher])


def _evaluate_candidate(candidate: bytes) -> float:
    score = 0.0

    for letter, freq in freqs.items():
        count = candidate.count(
            ord(letter)
        )  # don't even worry about counting uppercase letters (see note above)
        candidate_freq = count / len(candidate)
        err = abs(freq - candidate_freq)
        score += err

    return score


def _find_most_likely_message(_in: bytes) -> bytes:
    score = 99999
    ans = b""
    for trial_char in range(256):
        candidate = _single_char_xor(trial_char, _in)
        _score = _evaluate_candidate(candidate)
        if _score < score:
            score = _score
            ans = candidate

    return ans


def find_most_likely_message(_in: str) -> str:
    b_in = bytes.fromhex(_in)
    ans = _find_most_likely_message(b_in)
    return ans.decode()


_input = "1b37373331363f78151b7f2b783431333d78397828372d363c78373e783a393b3736"
assert find_most_likely_message(_input) == "Cooking MC's like a pound of bacon"

# Challenge 4


In [6]:
from pathlib import Path


def batch_find_most_likely_message(path: Path) -> bytes:
    answers: dict[bytes, float] = {}

    with path.open() as f:
        for line in f.read().splitlines():
            candidate = _find_most_likely_message(bytes.fromhex(line))
            answers[candidate] = _evaluate_candidate(candidate)

    return min(answers, key=answers.get)


assert (
    batch_find_most_likely_message(Path("./data/4.txt")).decode() == "Now that the party is jumping\n"
)

# Challenge 5


In [7]:
def repeating_key_xor(_in: bytes, key: bytes) -> bytes:
    n_reps = len(_in) // len(key) + 1  # extra 1 to cover gap from floor div
    rep_key = key * n_reps

    return bytes(
        [char ^ keychar for char, keychar in zip(_in, rep_key, strict=False)]
    )  # not strict, rep key might be longer


_in = b"""Burning 'em, if you ain't quick and nimble
I go crazy when I hear a cymbal"""

assert (
    repeating_key_xor(_in, b"ICE").hex()
    == "0b3637272a2b2e63622c2e69692a23693a2a3c6324202d623d63343c2a26226324272765272a282b2f20430a652e2c652a3124333a653e2b2027630c692b20283165286326302e27282f"
)